# 🚦 VAAET - Sistema Simple de Análisis de Tráfico

**Descripción**: Análisis de tráfico vehicular usando YOLOv11 para el Puente General Manuel Belgrano  
**Versión**: 2.1 - Simplificada

In [ ]:
# 📦 Instalación de dependencias
!pip install ultralytics opencv-python-headless psycopg2-binary tqdm --quiet

In [ ]:
# 🔧 Imports y configuración inicial
import cv2
import numpy as np
import os
import sys
import logging
import re
import math
import gc
from datetime import datetime, timedelta
from tqdm import tqdm
from typing import Dict, List, Optional, Tuple
from ultralytics import YOLO
import psycopg2

# Configuración básica
logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger('VAAET')

# Detectar entorno
IN_COLAB = 'google.colab' in sys.modules
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "No GPU"
except ImportError:
    GPU_AVAILABLE = False
    GPU_NAME = "PyTorch no disponible"

print("🚀 VAAET - Sistema de Análisis de Tráfico")
print(f"📍 Entorno: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"🖥️  GPU: {'✅ ' + GPU_NAME if GPU_AVAILABLE else '❌ No disponible'}")
print("✅ Configuración completada")

In [ ]:
# 🎯 Configuración de modelos y variables
YOLO_MODELS = {
    "nano": "yolo11n.pt",
    "small": "yolo11s.pt", 
    "medium": "yolo11m.pt",
    "large": "yolo11l.pt",
    "extra_large": "yolo11x.pt"
}

MODEL_CRITERIA = {
    "extra_large": 1.0,    # ≤ 1 hora
    "large": 3.0,          # 1-3 horas  
    "medium": 6.0,         # 3-6 horas
    "small": 12.0,         # 6-12 horas
    "nano": float('inf')   # > 12 horas
}

CONFIG = {
    'img_size': 640,
    'confidence': 0.5,
    'iou': 0.45,
    'parking_threshold': 2.0,
    'meters_per_pixel': 0.05,
    'ema_alpha': 0.2
}

# Variables globales
video_path = None
clip_id = None 
start_time_str = None
end_time_str = None
selected_model = None
enable_db = False

print("✅ Variables inicializadas")

In [ ]:
# 📁 Carga de video
def load_video():
    global video_path, clip_id, start_time_str, end_time_str
    
    if not IN_COLAB:
        print("⚠️  Función solo disponible en Google Colab")
        return False
        
    from google.colab import files
    
    print("📁 Selecciona tu video MP4:")
    uploaded = files.upload()
    
    if not uploaded:
        print("❌ No se subió ningún archivo")
        return False
        
    filename = list(uploaded.keys())[0]
    video_path = f"./{filename}"
    
    # Validar formato
    pattern = r"bridge_(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})_to_(\d{2}-\d{2}-\d{2})\.mp4"
    match = re.match(pattern, filename)
    
    if match:
        start_time_str, end_time_str = match.groups()
        clip_id = filename.replace(".mp4", "")
        print(f"✅ Video válido: {filename}")
    else:
        print("⚠️  Formato incorrecto. Ingrese fechas manualmente:")
        now = datetime.now()
        start_time_str = now.strftime("%Y-%m-%d_%H-%M-%S")
        end_time_str = (now + timedelta(hours=1)).strftime("%H-%M-%S")
        clip_id = f"bridge_{start_time_str}_to_{end_time_str}"
        print(f"📝 Usando: {clip_id}")
    
    return True

# Ejecutar carga
if load_video():
    print(f"🎥 Video listo: {clip_id}")

In [ ]:
# ⏱️ Análisis de duración y selección de modelo
def analyze_and_select_model():
    global selected_model
    
    if not video_path:
        print("❌ No hay video cargado")
        return False
        
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"❌ No se pudo abrir: {video_path}")
            return False
            
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration_hours = (frame_count / fps) / 3600 if fps > 0 else 0
        cap.release()
        
        # Seleccionar modelo
        selected_model = "nano"
        for model_name, max_hours in MODEL_CRITERIA.items():
            if duration_hours <= max_hours:
                selected_model = model_name
                break
                
        print(f"📊 Duración: {duration_hours:.2f} horas")
        print(f"🎯 Modelo seleccionado: {selected_model.upper()}")
        return True
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

# Ejecutar análisis
analyze_and_select_model()

In [ ]:
# 🗄️ Configuración de base de datos AWS RDS
def setup_database():
    global enable_db
    
    print("🗄️ CONFIGURACIÓN DE BASE DE DATOS AWS RDS")
    print("¿Persistir datos en PostgreSQL? (y/N): ")
    
    # Para demo: NO persistir por defecto
    enable_db = False  # Cambiar a True para habilitar BD
    
    if enable_db:
        print("✅ Base de datos HABILITADA")
        print("⚠️  Configure sus credenciales de AWS RDS en las variables de entorno:")
        print("   DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD")
    else:
        print("❌ Base de datos DESHABILITADA")
        print("📝 Los datos NO se persistirán")
    
    return enable_db

setup_database()

In [ ]:
# 🚗 Analizador de tráfico
class TrafficAnalyzer:
    def __init__(self, model_name):
        self.model = YOLO(YOLO_MODELS[model_name])
        self.tracks = {}
        self.records = []
        self.ema_speed = None
        
    def process_frame(self, frame, frame_idx, fps, timestamp):
        try:
            results = self.model.track(frame, persist=True, 
                                     imgsz=CONFIG['img_size'], 
                                     conf=CONFIG['confidence'],
                                     verbose=False)[0]
            
            if results.boxes is None:
                return frame
                
            boxes = results.boxes.xyxy.cpu().numpy()
            classes = results.boxes.cls.cpu().numpy()
            ids = results.boxes.id.cpu().numpy() if results.boxes.id is not None else []
            
            counts = {'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0, 'bicycle': 0}
            speeds = []
            
            for i, box in enumerate(boxes):
                cls_id = int(classes[i])
                class_name = self.model.names[cls_id]
                
                if class_name not in counts:
                    continue
                    
                counts[class_name] += 1
                x1, y1, x2, y2 = box
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                
                if i < len(ids):
                    track_id = int(ids[i])
                    if track_id in self.tracks:
                        prev_pos = self.tracks[track_id]['pos']
                        dx = cx - prev_pos[0]
                        dy = cy - prev_pos[1]
                        pixel_dist = math.sqrt(dx**2 + dy**2)
                        speed = (pixel_dist * fps * CONFIG['meters_per_pixel']) * 3.6
                        
                        if 0 < speed <= 100:
                            speeds.append(speed)
                            self.tracks[track_id] = {'pos': (cx, cy), 'speed': speed}
                        else:
                            self.tracks[track_id] = {'pos': (cx, cy), 'speed': 0}
                    else:
                        self.tracks[track_id] = {'pos': (cx, cy), 'speed': 0}
                
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                cv2.putText(frame, class_name, (int(x1), int(y1)-5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            if speeds:
                avg_speed = sum(speeds) / len(speeds)
                if self.ema_speed is None:
                    self.ema_speed = avg_speed
                else:
                    self.ema_speed = CONFIG['ema_alpha'] * avg_speed + (1 - CONFIG['ema_alpha']) * self.ema_speed
            
            h = frame.shape[0]
            cv2.putText(frame, f"Speed: {self.ema_speed:.1f} km/h" if self.ema_speed else "Speed: --", 
                       (10, h-20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            y = 30
            for vehicle, count in counts.items():
                cv2.putText(frame, f"{vehicle}: {count}", (10, y), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
                y += 25
            
            # Guardar para BD cada minuto
            if timestamp.minute != getattr(self, 'last_minute', None):
                self.last_minute = timestamp.minute
                if self.ema_speed and enable_db:
                    self.records.append({
                        'timestamp': timestamp.replace(second=0, microsecond=0),
                        'avg_speed': round(self.ema_speed, 2),
                        'counts': counts,
                        'total_vehicles': sum(counts.values())
                    })
            
            return frame
            
        except Exception as e:
            logger.error(f"Error procesando frame {frame_idx}: {e}")
            return frame

print("✅ Analizador definido")

In [ ]:
# 💾 Guardado en base de datos
def save_to_database(records, clip_id):
    if not enable_db or not records:
        print("⚠️  Base de datos deshabilitada o sin datos")
        return False
    
    try:
        # Configuración de BD (usar variables de entorno en producción)
        db_config = {
            "host": os.getenv('DB_HOST', 'bridge-traffic-db.cb2gcwmaimbx.sa-east-1.rds.amazonaws.com'),
            "port": int(os.getenv('DB_PORT', 5432)),
            "dbname": os.getenv('DB_NAME', 'postgres'),
            "user": os.getenv('DB_USER', 'postgres'),
            "password": os.getenv('DB_PASSWORD', 'your-secure-password')
        }
        
        conn = psycopg2.connect(**db_config)
        cur = conn.cursor()
        
        # Crear tabla si no existe
        cur.execute("""
            CREATE TABLE IF NOT EXISTS traffic_data (
                id SERIAL PRIMARY KEY,
                clip_id TEXT NOT NULL,
                record_time TIMESTAMP NOT NULL,
                avg_speed NUMERIC(5,2) NOT NULL,
                count_car INTEGER NOT NULL,
                count_truck INTEGER NOT NULL,
                count_bus INTEGER NOT NULL,
                count_motorcycle INTEGER NOT NULL,
                count_bicycle INTEGER NOT NULL,
                total_vehicles INTEGER NOT NULL,
                UNIQUE (clip_id, record_time)
            );
        """)
        
        # Insertar datos
        for record in records:
            cur.execute("""
                INSERT INTO traffic_data (
                    clip_id, record_time, avg_speed,
                    count_car, count_truck, count_bus,
                    count_motorcycle, count_bicycle, total_vehicles
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (clip_id, record_time) DO NOTHING;
            """, (
                clip_id, record['timestamp'], record['avg_speed'],
                record['counts']['car'], record['counts']['truck'], 
                record['counts']['bus'], record['counts']['motorcycle'], 
                record['counts']['bicycle'], record['total_vehicles']
            ))
        
        conn.commit()
        cur.close()
        conn.close()
        print(f"✅ {len(records)} registros guardados en PostgreSQL")
        return True
        
    except Exception as e:
        print(f"❌ Error guardando en BD: {e}")
        return False

print("✅ Función de BD lista")

In [ ]:
# 🚀 Procesamiento principal
def process_video():
    if not all([video_path, clip_id, start_time_str, selected_model]):
        print("❌ Faltan variables necesarias")
        return False
    
    try:
        print("🚀 Iniciando procesamiento...")
        
        # Cargar modelo
        print(f"📥 Cargando modelo {selected_model}...")
        analyzer = TrafficAnalyzer(selected_model)
        
        # Abrir video
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        # Video de salida
        output_path = f"VAAET_{clip_id}.mp4"
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        
        print(f"📊 Procesando {frame_count:,} frames...")
        start_datetime = datetime.strptime(start_time_str, "%Y-%m-%d_%H-%M-%S")
        
        # Procesar frames
        with tqdm(total=frame_count, desc="🚗 Analizando") as pbar:
            frame_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                timestamp = start_datetime + timedelta(seconds=frame_idx / fps)
                processed_frame = analyzer.process_frame(frame, frame_idx, fps, timestamp)
                out.write(processed_frame)
                
                frame_idx += 1
                pbar.update(1)
                
                if frame_idx % 1000 == 0:
                    gc.collect()
        
        cap.release()
        out.release()
        
        # Guardar en BD si está habilitada
        if enable_db:
            save_to_database(analyzer.records, clip_id)
        
        print("✅ Procesamiento completado!")
        print(f"🎥 Video generado: {output_path}")
        
        if enable_db:
            print(f"💾 Datos persistidos en PostgreSQL: {len(analyzer.records)} registros")
        else:
            print("📝 Datos NO persistidos (BD deshabilitada)")
        
        # Descargar video en Colab
        if IN_COLAB:
            from google.colab import files
            files.download(output_path)
            print("📥 Video descargado")
        
        return True
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

# ¡EJECUTAR PROCESAMIENTO!
print("🎯 EJECUTANDO VAAET...")
if process_video():
    print("🎉 ¡ANÁLISIS COMPLETADO!")
else:
    print("❌ Error en el análisis")